# 02 – Inntaksrammeverk: CSV → Bronze

Denne notebooken demonstrerer:
1. Last opp CSV-filer til `raw/`-bucketen i MinIO
2. Kjør inntaksjobb som leser CSV og skriver til `bronze/` som Delta-tabell
3. Verifiser resultatet og vis partisjonering
4. Bevis idempotens ved å re-kjøre jobben


## 1. Last opp testdata til MinIO (raw-lag)

Vi bruker `boto3` for å laste opp CSV-filene fra `/home/spark/notebooks/../data/sample/`.

In [1]:
import subprocess
subprocess.run(["pip", "install", "boto3", "-q"])

CompletedProcess(args=['pip', 'install', 'boto3', '-q'], returncode=0)

In [2]:
import boto3
from pathlib import Path

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="changeme",
)

sample_dir = Path("/home/spark/notebooks").parent / "data" / "sample"

for csv_file in sample_dir.glob("*.csv"):
    key = f"{csv_file.stem}/{csv_file.name}"
    s3.upload_file(str(csv_file), "raw", key)
    print(f"Lastet opp: s3a://raw/{key}")

ModuleNotFoundError: No module named 'boto3'

## 2. Kjør inntaksjobb: employees CSV → Bronze

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("02-ingest-csv-to-bronze")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 07:42:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
INGESTION_DATE = "2024-01-15"
SOURCE = "s3a://raw/employees"
TARGET = "s3a://bronze/employees"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{SOURCE}/*.csv")
)

df = df.withColumns({
    "ingestion_date": F.lit(INGESTION_DATE).cast("date"),
    "_source_path":   F.lit(SOURCE),
    "_ingested_at":   F.current_timestamp(),
})

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ingestion_date = '{INGESTION_DATE}'")
    .partitionBy("ingestion_date")
    .save(TARGET)
)

print(f"Skrev {df.count()} rader til {TARGET}")

26/03/28 07:42:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/28 07:42:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Skrev 8 rader til s3a://bronze/employees


## 3. Verifiser resultat

In [5]:
df_bronze = spark.read.format("delta").load(TARGET)

print(f"Rader i Bronze: {df_bronze.count()}")
df_bronze.show()
df_bronze.printSchema()

Rader i Bronze: 16
+---+-------+-----------+------+----------+--------------+-------------------+--------------------+
| id|   name| department|salary| hire_date|ingestion_date|       _source_path|        _ingested_at|
+---+-------+-----------+------+----------+--------------+-------------------+--------------------+
|  1|  Alice|engineering| 95000|2022-03-01|    2026-03-27|s3a://raw/employees|2026-03-28 00:00:...|
|  2|    Bob|  marketing| 72000|2021-07-15|    2026-03-27|s3a://raw/employees|2026-03-28 00:00:...|
|  3|Charlie|engineering|105000|2020-01-10|    2026-03-27|s3a://raw/employees|2026-03-28 00:00:...|
|  4|  Diana|  marketing| 68000|2023-05-20|    2026-03-27|s3a://raw/employees|2026-03-28 00:00:...|
|  5|    Eve|engineering| 98000|2019-11-03|    2026-03-27|s3a://raw/employees|2026-03-28 00:00:...|
|  6|  Frank|      sales| 61000|2023-09-01|    2026-03-27|s3a://raw/employees|2026-03-28 00:00:...|
|  7|  Grace|engineering|112000|2018-06-12|    2026-03-27|s3a://raw/employees|202

In [6]:
# Vis partisjoner
df_bronze.groupBy("ingestion_date").count().orderBy("ingestion_date").show()

+--------------+-----+
|ingestion_date|count|
+--------------+-----+
|    2024-01-15|    8|
|    2026-03-27|    8|
+--------------+-----+



## 4. Bevis idempotens — re-kjør med samme dato

`replaceWhere` erstatter partisjonen, ikke legger til. Antall rader skal ikke øke.

In [7]:
# Re-kjør nøyaktig samme skriving
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"ingestion_date = '{INGESTION_DATE}'")
    .partitionBy("ingestion_date")
    .save(TARGET)
)

antall_etter = spark.read.format("delta").load(TARGET).count()
print(f"Rader etter re-kjøring: {antall_etter}  ← skal fortsatt være 8")

Rader etter re-kjøring: 16  ← skal fortsatt være 8


In [8]:
# Delta-historikk viser begge skriveoperasjonene
from delta.tables import DeltaTable
DeltaTable.forPath(spark, TARGET).history().select("version", "timestamp", "operation").show()

+-------+-------------------+---------+
|version|          timestamp|operation|
+-------+-------------------+---------+
|      5|2026-03-28 07:42:58|    WRITE|
|      4|2026-03-28 07:42:33|    WRITE|
|      3|2026-03-28 00:00:40|    WRITE|
|      2|2026-03-27 06:49:38|    WRITE|
|      1|2026-03-27 06:39:49|    WRITE|
|      0|2026-03-25 17:48:05|    WRITE|
+-------+-------------------+---------+



In [10]:
spark.stop()